In [36]:
import sc2reader
import pandas as pd
import numpy as np
import tqdm
from glob import glob
from utils import *

all_dfs = []
game_number = 0
for account in ['sample']:
    files = glob(f'../data/input/my_data/{account}/*')
    for file in tqdm.tqdm(files):
        
        try:
            replay = sc2reader.load_replay(file, load_map=True)
        except Exception as e:
            print(f"Failed to load {file}: {e}")
            continue
        if replay.map_name in ['Ruby Rock LE', 'Emerald City CE', 'Reclamation LE', 'Fields of Death', 'Gemgarden LE', 'New Bed of Chaos LE', 'Rhoskallian LE', 'Rust Bucket LE', 'Sludge City', 'Undercurrent LE', 'Yellowjacket', 'Phantom Mode']:
            continue
        game_number += 1
        assert replay.map_name in ['valid_maps', "At Eternity's Edge LE", 'Blackrock LE', 'Fear and Faith LE', 'Rainfall LE', 'Sanctuary III LE', 'Lockdown LE', 'Washout LE', 'Rorschach LE', 'Old Sun Temple LE'], replay.map_name

        is_valid_release = replay.release_string >= '5.0.16'
        assert is_valid_release, replay.release_string
        
        player1 = 'nemo'
        player2 = None
        player1_won = None
        player1_race = 'Zerg'
        player2_race = None
        for player in replay.players:
            if player.name == 'nemo' or player.name == 'Kairo':
                assert player.play_race == 'Zerg'
                player1_won = player.result
            else:
                if player2 != None:
                    raise Exception(player.name, player2)
                player2 = player.name
                player2_race = player.play_race
        print(player1, player2, player1_race, player2_race)

        inject_times_data = []
        for event in replay.events:
            seconds = event.frame / 22.4
            minutes = int(seconds // 60)
            remaining_seconds = int(seconds % 60)
            if seconds < 210:
                continue
        
            try:
                if player2 in str(event):
                    continue
                if event.player == player2:
                    continue
            except:
                pass

            if 'SpawnLarva' in str(event):
                print(event)
            # print(seconds, event)
            elif event.name == 'TargetUnitCommandEvent':
                print(seconds, event)
                if event.ability == None:
                    continue
                if event.ability.name == 'SpawnLarva':
                    inject_times_data.append(seconds)
            if seconds > 215:
                break
        inject_times_df = pd.DataFrame()
        inject_times_df['inject_time'] = inject_times_data
        inject_times_df['game_number'] = game_number
        inject_times_df['opponent_race'] = player2_race
        inject_times_df['game_length_seconds_max_600'] = int(np.round(seconds,0))
        inject_times_df['main_player_won'] = player1_won
        inject_times_df = inject_times_df[['game_number', 'opponent_race', 'game_length_seconds_max_600', 'inject_time', 'main_player_won']]
        all_dfs.append(inject_times_df)
inject_times_df = pd.concat(all_dfs, axis='index', ignore_index=True)
# local.write.csv(inject_times_df, '3_extract_my_injects')

 33%|███▎      | 1/3 [00:01<00:02,  1.05s/it]

nemo Mandus Zerg Zerg


 67%|██████▋   | 2/3 [00:01<00:00,  1.05it/s]

nemo Red Zerg Terran
210.2232142857143 04.54	nemo            Right Click; Target: Hatchery [037C0001]; Location: (181.5, 107.5, 49104)
04.58	nemo            Ability (E20) - SpawnLarva; Target: Hive [03080001]; Location: (178.5, 73.5, 57296)


100%|██████████| 3/3 [00:03<00:00,  1.25s/it]

nemo TheReckoner Zerg Protoss
211.07142857142858 04.55	nemo            Ability (5C0) - Attack; Target: UnbuildablePlatesDestructible [00900001]; Location: (39.0, 43.0, 49104)


In [30]:
output_df



,unspent_time,unspent_amount,game_number
0,300.000000,300,1
1,307.142857,220,1
2,314.285714,235,1
3,321.428571,262,1
4,328.571429,238,1
5,335.714286,158,1
6,342.857143,170,1
7,350.000000,140,1
